# IMPORT

In [58]:
import pandas as pd
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt

import os
import gc
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

# DEFINE FOLDERS

In [59]:
BASE_FOLDER = Path("../data")
STATION_CLUSTER_FOLDER = BASE_FOLDER / "03_clustering" / "station_clusters"
DATASET_FOLDER = BASE_FOLDER / "01_clean" / "datasets"
MODEL_OUTPUT_FOLDER = BASE_FOLDER / "04_models" / "trained_models"
PLOT_FOLDER = BASE_FOLDER / "05_visualization" / "model_plots"
STATISTICS_FOLDER = BASE_FOLDER / "06_statistics" / "model_statistics"

folders = [STATION_CLUSTER_FOLDER, DATASET_FOLDER, MODEL_OUTPUT_FOLDER, PLOT_FOLDER, STATISTICS_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# DEFINE CONSTANTS

In [60]:
ALL_FEATURES = True

RANDOM_STATE = 42 # 42 or 72
K = 2
CLUSTER_METHOD = "kmeans"
FEATURE_NAME = "behavior+weather" if ALL_FEATURES else "behavior"
FILE_TAG = f"{CLUSTER_METHOD}_k{K}_feat_{FEATURE_NAME}_rs{RANDOM_STATE}"

CLUSTER_FILE_NAME = f"station_clusters_{FILE_TAG}"

HAS_BIKE_LAG = True
LAGGED_NAME = "lagged" if HAS_BIKE_LAG else "not_lagged"

MODEL_EXTENSION_TAG = f"feat_{FEATURE_NAME}_lag_{LAGGED_NAME}_rs{RANDOM_STATE}_k{K}"

# LOAD DATA AND CLUSTERS

In [61]:
df = pd.read_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/{CLUSTER_FILE_NAME}.csv")

df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)

Rows: (6710838, 21)
Clusters: cluster
0    3428850
1    3281988
Name: count, dtype: int64
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'total_bikes_lag1',
       'total_bikes_lag2', 'split', 'cluster'],
      dtype='object')


# FEATURE SELECTION

In [62]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Station context
    "capacity"
]

if ALL_FEATURES:
    FEATURES = FEATURES + [
                            # Weather
                            "temperature",
                            "wind_speed",
                            "wind_direction",
                            "weather_code",
                            "is_day",
                        ]
    
if HAS_BIKE_LAG:
    FEATURES = FEATURES + [
                            # Bike lag
                            "total_bikes_lag1", 
                            "total_bikes_lag2"
                        ]


TARGET = "total_bikes_now"


# TRAIN MODELS PER CLUSTER

In [63]:
pred_models = {}
results = []

for cluster_id in sorted(df["cluster"].unique()):
    print(f"\n - Training cluster {cluster_id}")

    # Select cluster and sort by time
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")

    if HAS_BIKE_LAG:
        if "total_bikes_lag1" not in df_c.columns:
            df_c["total_bikes_lag1"] = df_c.groupby("station_code")["total_bikes_now"].shift(1)
            df_c["total_bikes_lag2"] = df_c.groupby("station_code")["total_bikes_now"].shift(2)
            df_c = df_c.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

    # Select train/val using the split column
    train = df_c[df_c["split"] == 0]
    val   = df_c[df_c["split"] == 1]

    X_train, y_train = train[FEATURES], train[TARGET]
    X_val, y_val     = val[FEATURES], val[TARGET]

    models = {
        "LinearRegression": LinearRegression(),
        "SGDRegressor": Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ),
    }

    pred_models[cluster_id] = {}

    # Train and evaluate on validation set
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        r2 = r2_score(y_val, preds)
        mae = mean_absolute_error(y_val, preds)
        mse = mean_squared_error(y_val, preds)
        rmse = np.sqrt(mse)
        sse = mse * len(y_val)


        pred_models[cluster_id][name] = model

        results.append({
            "cluster": cluster_id,
            "model": name,
            "r2": r2,
            "mae": mae,
            "mse": mse,
            "rmse": rmse,
            "sse": sse,
            "n_samples": len(df_c)
        })

        print(f"  {name:16s} -> R^2 = {r2:.3f} | MAE = {mae:.2f} | MSE={mse:.2f} | RMSE={rmse:.2f} | SSE = {sse:.2f}")

        # Feature importance
        importances = None
        if name == "RandomForest":
            importances = model.feature_importances_
        elif name in ["LinearRegression", "SGDRegressor"]:
            # If pipeline, extract last step coefficients
            if isinstance(model, Pipeline):
                coef = model.named_steps["sgd"].coef_
            else:
                coef = model.coef_
            importances = np.abs(coef)
        else:
            importances = None

        if importances is not None:
            feat_imp_df = pd.DataFrame({
                "feature": FEATURES,
                "importance": importances
            }).sort_values("importance", ascending=False)

            # Plot
            plt.figure(figsize=(8, 5))
            plt.barh(feat_imp_df["feature"], feat_imp_df["importance"])
            plt.gca().invert_yaxis()
            plt.title(f"Feature Importance | Cluster {cluster_id} | {name} | {FEATURE_NAME} | {LAGGED_NAME}")
            plt.tight_layout()
            folder = Path(f"{PLOT_FOLDER}/{LAGGED_NAME}/{FEATURE_NAME}")
            folder.mkdir(parents=True, exist_ok=True)
            plt.savefig(f"{folder}/feature_importance_cluster_{cluster_id}_{name}.png")
            plt.close()
        else:
            print("NO PLOT!!!!")



 - Training cluster 0
  LinearRegression -> R^2 = 0.938 | MAE = 1.02 | MSE=2.97 | RMSE=1.72 | SSE = 1524213.34
  SGDRegressor     -> R^2 = 0.938 | MAE = 1.03 | MSE=2.97 | RMSE=1.72 | SSE = 1525889.59
  RandomForest     -> R^2 = 0.942 | MAE = 0.94 | MSE=2.75 | RMSE=1.66 | SSE = 1413990.01

 - Training cluster 1
  LinearRegression -> R^2 = 0.968 | MAE = 1.25 | MSE=4.59 | RMSE=2.14 | SSE = 2256356.16
  SGDRegressor     -> R^2 = 0.968 | MAE = 1.26 | MSE=4.59 | RMSE=2.14 | SSE = 2259162.68
  RandomForest     -> R^2 = 0.970 | MAE = 1.16 | MSE=4.31 | RMSE=2.08 | SSE = 2121227.34


# COMPARE RESULTS

In [64]:
results_df = pd.DataFrame(results)
results_df.sort_values(["cluster", "r2"], ascending=[True, False])

,cluster,model,r2,mae,mse,rmse,sse,n_samples
2,0,RandomForest,0.942313,0.937963,2.750739,1.658535,1.413990e+06,3428850
0,0,LinearRegression,0.937817,1.020913,2.965165,1.721965,1.524213e+06,3428850
1,0,SGDRegressor,0.937748,1.032019,2.968426,1.722912,1.525890e+06,3428850
5,1,RandomForest,0.969967,1.164755,4.311175,2.076337,2.121227e+06,3281988
3,1,LinearRegression,0.968054,1.253640,4.585810,2.141450,2.256356e+06,3281988
4,1,SGDRegressor,0.968014,1.259144,4.591514,2.142782,2.259163e+06,3281988


# GET BEST MODEL PER CLUSTER

In [65]:
best_models = {}

for cluster_id in results_df["cluster"].unique():
    best = results_df[results_df["cluster"] == cluster_id].sort_values(
        "r2", ascending=False
    ).iloc[0]

    best_models[cluster_id] = best["model"]

print(best_models)

{np.int64(0): 'RandomForest', np.int64(1): 'RandomForest'}


# SAVE ALL MODELS

In [66]:
for cluster_id, models_dict in pred_models.items():
    for name, model in models_dict.items():
        joblib.dump(model, f"{MODEL_OUTPUT_FOLDER}/{name}_model_cluster_{cluster_id}_{MODEL_EXTENSION_TAG}.pkl")

# QUICK TEST ON THE MODELS

In [67]:
def get_is_lagged(filename):
    return "not_lagged" not in filename

def get_has_behavior_weather(filename):
    return "behavior+weather" in filename

def get_features(filename):
    features = [
        # Time
        "sin_hour",
        "cos_hour",
        "is_weekend",

        # Station context
        "capacity"
    ]

    if get_has_behavior_weather(filename):
        features = features + [
                                # Weather
                                "temperature",
                                "wind_speed",
                                "wind_direction",
                                "weather_code",
                                "is_day",
                            ]
        
    if get_is_lagged(filename):
        features = features + [
                                # Bike lag
                                "total_bikes_lag1", 
                                "total_bikes_lag2"
                            ]
    return features

In [68]:
# LOAD THE MODELS
pred_models = {}

extension_tags = [
    f"feat_{fn}_lag_{lname}_rs{rs}_k{k}" 
    for fn in ["behavior+weather", "behavior"]
    for lname in ["lagged", "not_lagged"]
    for rs in [42]#[42, 72]
    for k in [2]
]

for cluster_id in range(2):
    pred_models[cluster_id] = {}
    for model_name in ["LinearRegression", "SGDRegressor", "RandomForest"]:
        pred_models[cluster_id][model_name] = []
        for extension_tag in extension_tags:
            filename = f"{MODEL_OUTPUT_FOLDER}/{model_name}_model_cluster_{cluster_id}_{extension_tag}.pkl"
            if os.path.exists(filename):
                pred_models[cluster_id][model_name].append({
                    "model": joblib.load(filename),
                    "is_lagged": get_is_lagged(filename),
                    "has_behavior_and_weather": get_has_behavior_weather(filename),
                    "features": get_features(extension_tag)
                    })
                
                print(f"Loaded: {filename}")
            else:
                print(f"File not found: {filename}")

Loaded: ..\data\04_models\trained_models/LinearRegression_model_cluster_0_feat_behavior+weather_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/LinearRegression_model_cluster_0_feat_behavior+weather_lag_not_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/LinearRegression_model_cluster_0_feat_behavior_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/LinearRegression_model_cluster_0_feat_behavior_lag_not_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/SGDRegressor_model_cluster_0_feat_behavior+weather_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/SGDRegressor_model_cluster_0_feat_behavior+weather_lag_not_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/SGDRegressor_model_cluster_0_feat_behavior_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/SGDRegressor_model_cluster_0_feat_behavior_lag_not_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_0_feat_behav

# PLOT IMPORTANCE OF MODELS

In [69]:
def plot_feature_importance(pred_models):
    for cluster_id in range(2):
        for model_name, versions in pred_models[cluster_id].items():
            for version_index, model_entry in enumerate(versions):
                model = model_entry["model"]
                is_lagged = model_entry["is_lagged"]
                has_behavior_and_weather = model_entry["has_behavior_and_weather"]
                features = model_entry["features"]
                lagged_name = "lagged" if is_lagged else "not_lagged"
                feature_name = "behavior+weather" if has_behavior_and_weather else "behavior"

                # Feature importance
                importances = None
                if model_name == "RandomForest":
                    importances = model.feature_importances_
                elif model_name in ["LinearRegression", "SGDRegressor"]:
                    # If pipeline, extract last step coefficients
                    if isinstance(model, Pipeline):
                        coef = model.named_steps["sgd"].coef_
                    else:
                        coef = model.coef_
                    importances = np.abs(coef)
                else:
                    importances = None

                if importances is not None:
                    feat_imp_df = pd.DataFrame({
                        "feature": features,
                        "importance": importances
                    }).sort_values("importance", ascending=False)

                    out_dir = PLOT_FOLDER / lagged_name / feature_name
                    out_dir.mkdir(parents=True, exist_ok=True)
                    # Plot
                    plt.figure(figsize=(8, 5))
                    plt.barh(feat_imp_df["feature"], feat_imp_df["importance"])
                    plt.gca().invert_yaxis()
                    plt.title(f"Feature Importance | Cluster {cluster_id} | {model_name} | {feature_name} | {lagged_name}")
                    plt.tight_layout()
                    plt.savefig(f"{PLOT_FOLDER}/feature_importance_cluster_{cluster_id}_{model_name}_v{version_index}.png")
                    plt.savefig(f"{out_dir}/feature_importance_cluster_{cluster_id}_{model_name}_v{version_index}.png")
                    plt.close()
                else:
                    print("NO PLOT!!!!")

In [70]:
plot_feature_importance(pred_models)

In [71]:
df_val = df[df["split"] == 1].copy()
del df
gc.collect()

79185

# VALIDATE

In [72]:
def plot_station(args):
    cluster_id, model_name, version_index, station_code, df_station = args
    
    model_entry = pred_models[cluster_id][model_name][version_index]
    model = model_entry["model"]
    features = model_entry["features"]
    is_lagged = model_entry["is_lagged"]
    has_behavior_and_weather = model_entry["has_behavior_and_weather"]

    df_station = df_station.sort_values("timestamp")

    X = df_station[features]
    y = df_station[TARGET]

    preds = model.predict(X)

    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mse = mean_squared_error(y, preds)
    rmse = np.sqrt(mse)
    sse = mse * len(y)

    
    """
    plt.figure(figsize=(14, 5))
    plt.plot(df_station["timestamp"], y, label="Actual", alpha=0.8)
    plt.plot(df_station["timestamp"], preds, label="Predicted", alpha=0.8)

    plt.title(
        f"Cluster {cluster_id} | Station {station_code}\n"
        f"Model: {model_name}"
    )
    plt.xlabel("Timestamp")
    plt.ylabel("Total Bikes Now")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        f"{PLOT_FOLDER}/cluster_{cluster_id}_station_{station_code}_{FEATURE_NAME}_{LAGGED_NAME}.png"
    )
    plt.close()
    """

    return {
        "cluster": cluster_id,
        "station": station_code,
        "model": model_name,
        "is_lagged": is_lagged,
        "has_behavior_and_weather": has_behavior_and_weather,
        "r2": r2,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "sse": sse,
        "n_samples": len(df_station),
    }


In [73]:
os.cpu_count()

16

In [74]:
jobs = [
    (cluster_id, model_name, version_index, station_code, df_station)
    for (cluster_id, station_code), df_station in df_val.groupby(["cluster", "station_code"], sort=False)
    for model_name, version_index in pred_models[cluster_id].items()
    for version_index in range(len(pred_models[cluster_id][model_name]))
]
results = []

with ThreadPoolExecutor(max_workers=6) as ex:
    for res in tqdm(ex.map(plot_station, jobs), total=len(jobs)):
        results.append(res)

results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{STATISTICS_FOLDER}/all_stations_results_all_models.csv",
    index=False
)

100%|██████████| 17004/17004 [05:59<00:00, 47.28it/s]
